Testing data retrieval from BasketballReference.com

In [3]:
from io import StringIO
import pandas as pd
from bs4 import BeautifulSoup
import requests
import numpy as np

In [4]:
# URL get fuction omitted

In [5]:
def get_regular_season_stats(url: str) -> pd.DataFrame:
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    table_html = soup.find('table', {'id': 'per_game_stats'})
    
    if table_html:
        return pd.read_html(StringIO(str(table_html)))[0]
    else:
        print("Regular season stats table not found.")
        return pd.DataFrame()

def get_playoff_stats(url: str) -> pd.DataFrame:
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    table_html = soup.find('table', {'id': 'per_game_stats_post'})
    
    if table_html:
        return pd.read_html(StringIO(str(table_html)))[0]
    else:
        print("Playoff stats table not found.")
        return pd.DataFrame()

In [7]:
URL = "https://www.basketball-reference.com/players/b/butleji01.html"
reg = get_regular_season_stats(URL)
post = get_playoff_stats(URL)

In [8]:
# data[1], data[2] represent the regular season/playoffs stats
# although not visible normally, the data is stored in the page when loaded,
# thus there is no need for Selenium

In [9]:
reg.head(2)

,Season,Age,Team,Lg,Pos,G,GS,MP,FG,FGA,...,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,Awards
0,2011-12,22,CHI,NBA,SG,42.0,0.0,8.5,0.8,1.9,...,0.5,0.8,1.3,0.3,0.3,0.1,0.3,0.5,2.6,NaN


In [11]:
post.head(2)

,Season,Age,Team,Lg,Pos,G,GS,MP,FG,FGA,...,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,Awards
0,2011-12,22,CHI,NBA,SG,3.0,0.0,1.3,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.3,0.0,NaN
1,2012-13,23,CHI,NBA,SF,12.0,12.0,40.8,4.2,9.6,...,0.8,4.4,5.2,2.7,1.3,0.5,1.3,2.2,13.3,NaN


We can see this works well, NaN data is when a player didn't play that year. Ex: in the 2015-2016 season, Jimm Butler
did not make the playoffs. We will clean this later

Need to ensure behaviour is consistent among all player examples. Further testing will be done with other players

In [12]:
URL = "https://www.basketball-reference.com/players/d/duranke01.html"
reg = get_regular_season_stats(URL)
post = get_playoff_stats(URL)

In [13]:
reg.head(2)

,Season,Age,Team,Lg,Pos,G,GS,MP,FG,FGA,...,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,Awards
0,2007-08,19,SEA,NBA,SG,80,80,34.6,7.3,17.1,...,0.9,3.5,4.4,2.4,1.0,0.9,2.9,1.5,20.3,ROY-1
1,2008-09,20,OKC,NBA,SF,74,74,39.0,8.9,18.8,...,1.0,5.5,6.5,2.8,1.3,0.7,3.0,1.8,25.3,NaN


In [14]:
post.head(2)

,Season,Age,Team,Lg,Pos,G,GS,MP,FG,FGA,...,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,Awards
0,2009-10,21,OKC,NBA,SF,6.0,6.0,38.5,7.2,20.5,...,1.3,6.3,7.7,2.3,0.5,1.3,3.7,2.8,25.0,NaN
1,2010-11,22,OKC,NBA,SF,17.0,17.0,42.5,9.1,20.3,...,1.1,7.1,8.2,2.8,0.9,1.1,2.5,3.1,28.6,NaN


Raw data works well, will do cleaning later

## CLEANING

In [15]:
def convert_type(df: pd.DataFrame) -> pd.DataFrame:
    numerical_columns = ["G", "PTS", "TRB", "AST", "STL", "BLK"]
    for col in numerical_columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df

df = convert_type(reg)
# df.head()

In [16]:
df.columns

Index(['Season', 'Age', 'Team', 'Lg', 'Pos', 'G', 'GS', 'MP', 'FG', 'FGA',
       'FG%', '3P', '3PA', '3P%', '2P', '2PA', '2P%', 'eFG%', 'FT', 'FTA',
       'FT%', 'ORB', 'DRB', 'TRB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS',
       'Awards'],
      dtype='object')

In [18]:
newdf = df.drop(['Age', 'Lg', 'GS', 'FG%', '3P', '3PA', '3P%', '2P', '2PA', '2P%', 'eFG%', 
             'FT', 'FTA', 'FT%', 'ORB', 'DRB', 'TOV', 'PF','Awards'], axis=1)

In [19]:
newdf.head(2)

,Season,Team,Pos,G,MP,FG,FGA,TRB,AST,STL,BLK,PTS
0,2007-08,SEA,SG,80.0,34.6,7.3,17.1,4.4,2.4,1.0,0.9,20.3
1,2008-09,OKC,SF,74.0,39.0,8.9,18.8,6.5,2.8,1.3,0.7,25.3


In [20]:
def clean_table(df: pd.DataFrame) -> pd.DataFrame:
    df = convert_type(df)
    df = df.dropna(axis=0, how='all')
    df.index = range(1, len(df) + 1)
#     clean_aggregates(df)
    return df

In [21]:
newdf = clean_table(newdf)
newdf.head(2)

,Season,Team,Pos,G,MP,FG,FGA,TRB,AST,STL,BLK,PTS
1,2007-08,SEA,SG,80.0,34.6,7.3,17.1,4.4,2.4,1.0,0.9,20.3
2,2008-09,OKC,SF,74.0,39.0,8.9,18.8,6.5,2.8,1.3,0.7,25.3


In [22]:
has_yrs = newdf['Season'].str.contains('Yrs', case=False)
has_yrs.head()

1    False
2    False
3    False
4    False
5    False
Name: Season, dtype: object

In [54]:
# newdf[has_yrs].index.tolist()

In [25]:
newdf.head(2)

,Season,Team,Pos,G,MP,FG,FGA,TRB,AST,STL,BLK,PTS
1,2007-08,SEA,SG,80.0,34.6,7.3,17.1,4.4,2.4,1.0,0.9,20.3
2,2008-09,OKC,SF,74.0,39.0,8.9,18.8,6.5,2.8,1.3,0.7,25.3


data cleaning done below

In [26]:
def convert_type(df):
    # Define the numerical columns you want to convert
    numerical_columns = ["G", "PTS", "TRB", "AST", "STL", "BLK"]
    
    # Use .copy() to avoid SettingWithCopyWarning
    df = df.copy()
    
    # Loop through each column and convert to numeric if it exists
    for col in numerical_columns:
        if col in df.columns:  # Check if the column exists
            df[col] = pd.to_numeric(df[col], errors="coerce")  # Convert to numeric
        else:
            print(f"Warning: Column '{col}' does not exist in the DataFrame.")

    return df

def clean_aggregates(df: pd.DataFrame):
    # create boolean Series
    has_yrs = df['Season'].str.contains('Yr', case=False)
    yrs_indices = df[has_yrs].index.tolist()
    agg_count = 1

    for i in yrs_indices:
        # entire career case
        if '(' not in df.at[i, 'Season']:
            df.at[i, 'Season'] = 'Career'
        # one team case
        else:
            team = df.at[i, 'Season']
            pos = team.find('(')
            df.at[i, 'Season'] = team[:pos]
            df.at[i, 'Team'] = np.nan

        df.rename(index={i: f"TOT_{agg_count}"}, inplace=True)
        agg_count += 1
    
def clean_fill(df: pd.DataFrame):
    df.replace(r"^Did not play.*", "-", regex=True, inplace=True)
    df.fillna(-1, inplace=True)
    df['G'] = df['G'].astype(int)
    
def clean_table(df: pd.DataFrame) -> pd.DataFrame:

    if df.empty:
        return df # must fix later
     
    df = convert_type(df)
#     df = df.drop(['Age', 'Lg', 'GS', 'FG%', '3P', '3PA', '3P%', '2P', '2PA', '2P%', 'eFG%', 
#              'FT', 'FTA', 'FT%', 'ORB', 'DRB', 'TOV', 'PF','Awards'], axis=1)
    df = df.dropna(axis=0, how='all')
    df.index = range(1, len(df) + 1)
    clean_aggregates(df)
    clean_fill(df)
    return df

In [27]:
# df = clean_table(newdf)
# df

## Functions

In [28]:
# Define a function to exclude unwanted items
def is_actual_team(team):
    # Convert to string to ensure consistent comparison
    team_str = str(team)
    # Exclude entries with '-1', '-', '2TM', or containing "Yrs"
    return team_str not in ['-1', '-', '2TM'] and 'Yrs' not in team_str

# Apply the filter to keep only actual team names
teams = df['Team'].value_counts().index.tolist()
sig = [team for team in teams if is_actual_team(team)]
sig

['OKC', 'GSW', 'BRK', 'PHO', 'SEA', 'Did not play - injury']

In [29]:
sig[:len(sig)//2]

['OKC', 'GSW', 'BRK']

In [30]:
wilt = "https://www.basketball-reference.com/players/c/chambwi01.html"
wiltreg = get_regular_season_stats(wilt)
wiltreg.head(2)

,Season,Age,Team,Lg,Pos,G,MP,FG,FGA,FG%,FT,FTA,FT%,TRB,AST,PF,PTS,Awards
0,1959-60,23,PHW,NBA,C,72,46.4,14.8,32.1,.461,8.0,13.8,.582,27.0,2.3,2.1,37.6,"MVP-1,ROY-1,AS,NBA1"
1,1960-61,24,PHW,NBA,C,79,47.8,15.8,31.1,.509,6.7,13.3,.504,27.2,1.9,1.6,38.4,"MVP-4,AS,NBA1"


In [32]:
wiltreg2 = clean_table(wiltreg)
wiltreg2.head(2)

,Season,Age,Team,Lg,Pos,G,MP,FG,FGA,FG%,FT,FTA,FT%,TRB,AST,PF,PTS,Awards
1,1959-60,23,PHW,NBA,C,72,46.4,14.8,32.1,.461,8.0,13.8,.582,27.0,2.3,2.1,37.6,"MVP-1,ROY-1,AS,NBA1"
2,1960-61,24,PHW,NBA,C,79,47.8,15.8,31.1,.509,6.7,13.3,.504,27.2,1.9,1.6,38.4,"MVP-4,AS,NBA1"
